In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
print(train_df.shape, test_df.shape)


(699960, 26) (299984, 25)


In [ ]:
train_df["is_fraud"].value_counts()

is_fraud
0.0    688271
1.0     11689
Name: count, dtype: int64

Data Preprocessing

In [ ]:
test_ids = test_df["id"]


y = train_df["is_fraud"].astype(int)
X = train_df.drop(columns=["id", "is_fraud"])
X_test = test_df.drop(columns=["id"])

In [ ]:
bool_cols = [
    "is_foreign_transaction", "is_new_merchant",
    "used_vpn", "ip_country_mismatch", "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt"
]
X[bool_cols] = X[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [ ]:
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

encoded_train = encoder.fit_transform(X[cat_cols])
encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X.index
)

encoded_test = encoder.transform(X_test[cat_cols])
encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X_test.index
)

X = pd.concat([X.drop(columns=cat_cols), encoded_train_df], axis=1)
X_test = pd.concat([X_test.drop(columns=cat_cols), encoded_test_df], axis=1)

print(X.shape, X_test.shape)

(699960, 54) (299984, 54)


**XGBOOST**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

print(X_train.shape, X_val.shape)

(559968, 54) (139992, 54)


In [ ]:
model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=1,
    eval_metric="auc"
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=25)

[0]	validation_0-auc:0.88464
[25]	validation_0-auc:0.93938
[50]	validation_0-auc:0.94258
[75]	validation_0-auc:0.94355
[99]	validation_0-auc:0.94394


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
val_pred = model.predict_proba(X_val)[:, 1]
print("ROC AUC:", roc_auc_score(y_val, val_pred))

ROC AUC: 0.9439394858084011


In [ ]:
test_pred = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"id": test_ids, "is_fraud": test_pred})
submission.to_csv("submission.csv", index=False)
submission.head()

,id,is_fraud
0,699960,0.000351
1,699961,0.000243
2,699962,0.059604
3,699963,0.002162
4,699964,0.001771
